
### Load data
Ransomware.live has no single download file, so I pull the data from its public API. The `/victims/<year>/<month>` endpoint returns the victims for one month, so the code below loops over every month from 2020 to today, waits one second between requests (the docs say usage may be rate-limited), and saves the raw result to `ransomware_live_raw.json`. Re-running the notebook then reads that file instead of calling the API again, which also gives us a fixed **snapshot**, because the live data keeps changing.

In [2]:
import json
import time
from pathlib import Path

%pip install pycountry

import altair as alt
import pandas as pd
import pycountry
import requests

alt.data_transformers.disable_max_rows()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

Note: you may need to restart the kernel to use updated packages.


In [3]:
BASE_URL = "https://api.ransomware.live/v2"
START_YEAR = 2021                      
RAW_FILE = Path("ransomware_live_raw.json")


def get_json(path, retries=5):
    '''GET one API endpoint. Back off and retry if the API answers 429 (rate limit hit).'''
    for attempt in range(retries):
        r = requests.get(f"{BASE_URL}/{path}", timeout=60)
        if r.status_code == 200:
            data = r.json()
            if isinstance(data, dict):            
                data = data.get("victims") or data.get("data") or []
            return data
        if r.status_code == 404:                
            return []
        if r.status_code == 429:
            time.sleep(4 * 2 ** attempt)         
            continue
        r.raise_for_status()
    raise RuntimeError(f"Gave up on {path} after {retries} tries")


def download_victims(start_year):
    today = pd.Timestamp.today()
    rows = []
    for year in range(start_year, today.year + 1):
        last_month = today.month if year == today.year else 12
        for month in range(1, last_month + 1):
        
            batch = get_json(f"victims/{year}/{month}") or get_json(f"victims/{year}/{month:02d}")
            for rec in batch:
                rec["requested_month"] = f"{year}-{month:02d}"
            rows.extend(batch)
            print(f"{year}-{month:02d}: {len(batch):>5} victims")
            time.sleep(1)                          
    return rows


if RAW_FILE.exists():
    rows = json.loads(RAW_FILE.read_text())
    print(f"Loaded cached snapshot: {len(rows):,} records")
else:
    rows = download_victims(START_YEAR)
    RAW_FILE.write_text(json.dumps(rows))
    print(f"Saved snapshot: {len(rows):,} records")

df = pd.DataFrame(rows)

Loaded cached snapshot: 31,095 records


### Describe

In [4]:
df.head()

,activity,attackdate,claim_url,country,data_size,description,discovered,domain,group,press,ransom,screenshot,url,victim,requested_month
0,Agriculture and Food Production,2022-04-30T11:26:52.208956+00:00,,,None,,2022-04-30T11:26:52.208956+00:00,sagefruit.com,lockbit2,None,None,,https://www.ransomware.live/id/c2FnZWZydWl0LmN...,sagefruit.com,2022-04
1,Not Found,2022-04-30T11:26:51.593497+00:00,,,None,,2022-04-30T11:26:51.593497+00:00,,lockbit2,None,None,,https://www.ransomware.live/id/cEBsb2NrYml0Mg==,p,2022-04
2,Not Found,2022-04-30T11:26:50.541750+00:00,,,None,,2022-04-30T11:26:50.541750+00:00,,lockbit2,None,None,,https://www.ransomware.live/id/aGlzcGFub2FtZXJ...,hispanoamerican...,2022-04
3,Other,2022-04-30T11:26:49.248196+00:00,,,None,,2022-04-30T11:26:49.248196+00:00,,lockbit2,None,None,,https://www.ransomware.live/id/N2dlbmVyYXRpb25...,7generations.or...,2022-04
4,Not Found,2022-04-30T03:42:20.417343+00:00,,,None,,2022-04-30T03:42:20.417343+00:00,,lockbit2,None,None,,https://www.ransomware.live/id/d0Bsb2NrYml0Mg==,w,2022-04


In [5]:
print(f"Rows: {len(df):,}    Columns: {df.shape[1]}")
print("Columns:", list(df.columns))

nested = [c for c in df.columns if df[c].map(lambda v: isinstance(v, (dict, list))).any()]
print("\nNested columns:", nested)

df.info()

Rows: 31,095    Columns: 15
Columns: ['activity', 'attackdate', 'claim_url', 'country', 'data_size', 'description', 'discovered', 'domain', 'group', 'press', 'ransom', 'screenshot', 'url', 'victim', 'requested_month']

Nested columns: ['press']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31095 entries, 0 to 31094
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   activity         31095 non-null  object
 1   attackdate       31095 non-null  object
 2   claim_url        31095 non-null  object
 3   country          31095 non-null  object
 4   data_size        600 non-null    object
 5   description      31095 non-null  object
 6   discovered       31095 non-null  object
 7   domain           31095 non-null  object
 8   group            31095 non-null  object
 9   press            1552 non-null   object
 10  ransom           145 non-null    object
 11  screenshot       31095 non-null  object
 12  url        

#### Describing the data

The data comes from Ransomware.live, a project by security researcher Julien Mousqueton that monitors the leak sites where ransomware groups name the organizations they have attacked. We downloaded it from the free v2 API on Nov 16, 2026, one month at a time.

Each row is one claim, a victim organization listed by a ransomware group on its leak site. The same organization can appear more than once. As an example, claimed by two groups or re-posted.

For the columns, the core fields are organization name which are victims, group (the ransomware group that claimed it), attackdate is date of the incident and country (ISO-2 code, such as US or DE). The API also returns nested fields that we leave out of the analysis table. Other columns in our download. There are 15 rows and 3 columns before cleaning.

#### Challenges
1. A leak site only shows victims that the group chose to name. Victims who paid quietly, and groups that do not run leak sites, are invisible. Counts are a lower bound, not all ransomware attacks. 
2. Ransomware.live keeps adding groups and sites to its monitoring, so part of any upward trend may reflect better tracking rather than more attacks.
3. Attackdate may be an estimate or the posting date, and the month endpoint may be organized by when a victim was discovered. We test this below.
4. Country and sector are not always filled in.
5. The same victim can appear several times.
6. The numbers change daily, so we cache a snapshot and cite the access date.
7. Victim names are real organizations, many of them hospitals, schools or small businesses. We aggregate before plotting and do not put names in public charts.
8. The site has a Terms & Conditions page. We cite the source and use the data for coursework only.

### EDA

In [6]:
rw = df.drop(columns=nested).copy()
rw.columns = rw.columns.str.strip().str.lower()

placeholders = {"", "not found", "unknown", "n/a", "none"}
for c in rw.select_dtypes(include=["object", "string"]).columns:
    s = rw[c].astype("string").str.strip()
    rw[c] = s.mask(s.str.lower().isin(placeholders))

print("Share of missing values per column (%):")
print((rw.isna().mean() * 100).round(1).sort_values(ascending=False).head(10))

Share of missing values per column (%):
ransom         99.5
data_size      98.1
screenshot     32.3
claim_url      27.3
country        19.7
domain         14.9
description    14.6
activity        4.5
attackdate      0.0
discovered      0.0
dtype: float64


In [7]:
def to_date(series):
    '''Parse mixed date strings to timezone-free datetimes; unparseable values become NaT.'''
    try:
        parsed = pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
    except (TypeError, ValueError):                
        parsed = pd.to_datetime(series, errors="coerce", utc=True)
    return parsed.dt.tz_localize(None)


rw["attack_date"] = to_date(rw["attackdate"])
if "discovered" in rw.columns:
    rw["discovered_date"] = to_date(rw["discovered"])

n0 = len(rw)

dupe_key = [c for c in ["victim", "group", "attack_date"] if c in rw.columns]
rw = rw.drop_duplicates(subset=dupe_key)
print(f"Duplicate rows removed:      {n0 - len(rw):,}")

ok = rw["attack_date"].between(pd.Timestamp("2013-01-01"), pd.Timestamp.today())
print(f"Bad or missing dates removed: {(~ok).sum():,}")
rw = rw[ok].copy()

rw["year"] = rw["attack_date"].dt.year
rw["year_month"] = rw["attack_date"].dt.to_period("M").dt.to_timestamp()
print(f"Rows kept: {len(rw):,}  (from {n0:,})")
print(f"Attack dates run from {rw['attack_date'].min():%Y-%m-%d} to {rw['attack_date'].max():%Y-%m-%d}")

Duplicate rows removed:      1
Bad or missing dates removed: 0
Rows kept: 31,094  (from 31,095)
Attack dates run from 2021-12-15 to 2026-11-16


In [8]:
iso2_to_name = {c.alpha_2: c.name for c in pycountry.countries}
rw["country_code"] = rw["country"].str.upper().replace({"UK": "GB"})
unknown_codes = sorted(set(rw.loc[rw["country_code"].notna() & ~rw["country_code"].isin(iso2_to_name), "country_code"]))
print("Codes that are not valid ISO-2 (set to missing):", unknown_codes)
rw.loc[~rw["country_code"].isin(iso2_to_name), "country_code"] = pd.NA
rw["country_name"] = rw["country_code"].map(iso2_to_name)

sector_col = next((c for c in ["activity", "sector"] if c in rw.columns), None)
if sector_col:
    rw["sector"] = rw[sector_col]
    print(f"Sector column found: '{sector_col}' ({rw['sector'].nunique()} distinct values)")
else:
    print("No sector column in this download. Check df.columns or /sectorvictims/<sector>.")

print(f"\nRows with a country: {rw['country_code'].notna().mean():.1%}")
if sector_col:
    print(f"Rows with a sector:  {rw['sector'].notna().mean():.1%}")

Codes that are not valid ISO-2 (set to missing): []
Sector column found: 'activity' (13 distinct values)

Rows with a country: 80.3%
Rows with a sector:  95.5%


In the above steps I have done tidy column names and placeholder text, parse dates, dropped duplicates and impossible dates, convert country codes to names, and find the sector column.

In [9]:
if "requested_month" in rw.columns:
    share = (rw["attack_date"].dt.strftime("%Y-%m") == rw["requested_month"]).mean()
    print(f"{share:.1%} of records have an attack date in the month they were requested under")

if "discovered_date" in rw.columns:
    lag = (rw["discovered_date"] - rw["attack_date"]).dt.days
    print("\nDays between attack date and discovery date:")
    print(lag.describe().round(1))

97.4% of records have an attack date in the month they were requested under

Days between attack date and discovery date:
count    31094.0
mean        19.7
std         78.7
min      -1144.0
25%          0.0
50%          0.0
75%          7.0
max       1211.0
dtype: float64


The month endpoint follows attackdate closely: 97.4% of records have an attack date in the month they were requested under, so grouping by attackdate is safe for our monthly and yearly counts. The gap between the attack date and the discovery date is short for most victims. Across the 31,081 records that have both dates, the median gap is 0 days and 75% of records are within 7 days. The average (19.7 days) is much higher than the median because of a few extreme values (standard deviation 78.7 days), with the largest gap being 1,211 days, a victim posted more than three years after the attack. The minimum is -1,144 days, which means a victim was "discovered" about three years before its attack date. This is impossible and points to data errors or back-dated entries. Because at least a quarter of records have a gap of exactly 0 days, attackdate is often probably the same as the date the victim appeared on the leak site rather than the true date of the attack. We therefore keep attackdate as our time variable but treat our monthly trends as approximate, showing when victims were publicly named.

#### Exploratory charts of the Ransomware.live data
Three charts of the cleaned table rw, one for each question we can later compare with EuRepoC: when victims were posted, where they are, and which sectors they belong to. The same three questions are the ones the two datasets share (dates, countries and sectors or target categories).

Two choices apply to all three charts:
- The API returned no victims before April 2022, and the current month is still in progress. We therefore chart complete months only, from April 2022 up to last month.
- All charts use the Ransomware.live red that our group uses for this dataset.

In [10]:
RW_COLOR = "#e45756"        

first_month = pd.Timestamp("2022-04-01")                      
this_month = pd.Timestamp.today().to_period("M").to_timestamp()  
rw_eda = rw[(rw["year_month"] >= first_month) & (rw["year_month"] < this_month)].copy()

print(f"{len(rw_eda):,} of {len(rw):,} victims fall in complete months "
      f"({rw_eda['year_month'].min():%b %Y} to {rw_eda['year_month'].max():%b %Y})")

30,546 of 31,094 victims fall in complete months (Apr 2022 to Oct 2026)


**Chart 1. When were victims posted?** 

In [11]:
monthly = rw_eda.groupby("year_month").size().rename("victims").reset_index()
first, last = monthly["year_month"].min(), monthly["year_month"].max()

alt.Chart(monthly).mark_line(color=RW_COLOR, point=alt.OverlayMarkDef(color=RW_COLOR)).encode(
    x=alt.X("year_month:T", title="Month", axis=alt.Axis(format="%Y", tickCount="year")),
    y=alt.Y("victims:Q", title="Victims posted"),
    tooltip=[alt.Tooltip("year_month:T", title="Month", format="%B %Y"),
             alt.Tooltip("victims:Q", title="Victims", format=",")]
).properties(title=f"Ransomware Victims Posted per Month ({first:%b %Y} to {last:%b %Y})",
             width=650, height=300)

alt.Chart(...)

How many victims were posted each month, and is the number growing? That part of any growth can come from Ransomware.live monitoring more groups over time (challenge 2), and that attackdate is often the date the victim was posted.

**Chart 2. Where are the victims?**

In [12]:
known = rw_eda.dropna(subset=["country_name"])
top_countries = (known["country_name"].value_counts().head(10)
                      .rename_axis("country").reset_index(name="victims"))

bars = alt.Chart(top_countries).mark_bar(color=RW_COLOR).encode(
    x=alt.X("victims:Q", title="Number of victims", axis=alt.Axis(tickCount=6),
            scale=alt.Scale(domainMax=top_countries["victims"].max() * 1.12)),
    y=alt.Y("country:N", sort="-x", title=None),
    tooltip=["country:N", alt.Tooltip("victims:Q", format=",")])
labels = bars.mark_text(align="left", dx=3).encode(text=alt.Text("victims:Q", format=","))

(bars + labels).properties(
    title=alt.TitleParams("Top 10 Countries by Ransomware Victims",
                          subtitle=f"Only the {len(known) / len(rw_eda):.0%} of victims that have a country"),
    width=500)

alt.LayerChart(...)

Only victims with a country are counted, and the title shows what share that is. These are raw counts, so large countries are expected to lead. The World Bank table (iso2, population_millions) will let us turn this into victims per million people.

**Chart 3. Which sectors are hit, and did the mix change?** 

In [13]:
sec = rw_eda.dropna(subset=["sector"])
sec_year = sec.groupby(["year", "sector"]).size().rename("victims").reset_index()
sec_year["share"] = sec_year["victims"] / sec_year.groupby("year")["victims"].transform("sum")
sector_order = sec["sector"].value_counts().index.tolist()
dark = (sec_year["share"].min() + sec_year["share"].max()) / 2   # cells above the midpoint get white text

cells_ = alt.Chart(sec_year).mark_rect().encode(
    x=alt.X("year:O", title="Year", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("sector:N", sort=sector_order, title=None),
    color=alt.Color("share:Q", scale=alt.Scale(scheme="reds"),
                    legend=alt.Legend(title="Share of year", format=".0%")),
    tooltip=["year:O", "sector:N", alt.Tooltip("victims:Q", format=","), alt.Tooltip("share:Q", format=".1%")])
numbers = alt.Chart(sec_year).mark_text(fontSize=10).encode(
    x="year:O", y=alt.Y("sector:N", sort=sector_order),
    text=alt.Text("share:Q", format=".0%"),
    color=alt.condition(f"datum.share > {dark}", alt.value("white"), alt.value("black")))

(cells_ + numbers).properties(
    title=alt.TitleParams("Share of Each Year's Victims by Sector",
                          subtitle=f"Only the {len(sec) / len(rw_eda):.0%} of victims that have a sector"),
    width=330, height=330)

alt.LayerChart(...)

For each year, what share of victims belongs to each sector? Shares (not counts) are used because 2022 (April to December) and 2026 (January to last month) are partial years, and the sector label is missing for a small part of the data. Sectors are sorted from most to least common overall.


### Summary

Ransomware.live is a large, fast-growing list of criminal victims with a date, a country and a sector. It is the volume side of the story. EuRepoC is the opposite: a smaller list of *politically significant* incidents with rich detail about who attacked whom. Together they let us ask whether the two kinds of attacks follow the same geography and timeline.

**Patterns and relationships worth investigating**
1. **Trend over time.** Do monthly ransomware victims rise after 2020, and does the jump in EuRepoC incidents against the US (about 70 in 2022 to about 236 in 2023 in our teammate's chart) coincide with it? We must keep in mind that part of the growth is better monitoring (challenge 2).
2. **Geography.** Which countries have the most victims, and does that change once we divide by population? Are those the same countries that appear most often as targets in EuRepoC?
3. **Do the two datasets move together?** A country-level correlation between EuRepoC incidents and ransomware victims, and a direct comparison of EuRepoC's own *Ransomware* incidents (about 210 against the US) with the far larger number of leak-site victims. The gap shows how little of ransomware reaches the "politically significant" list.
4. **Concentration among groups.** Are a handful of groups responsible for most victims, and how quickly do top groups appear and disappear?
5. **Sectors.** Which sectors does ransomware hit compared with EuRepoC's receiver categories (critical infrastructure, state institutions, corporate targets)?

It needs more data we can use from below:
- **Population** (World Bank, added in 2d) to normalize country counts. A possible fourth source is the ITU Global Cybersecurity Index, to test whether countries with higher cybersecurity scores see fewer victims.
- A hand-made **crosswalk** between Ransomware.live's sector labels and EuRepoC's `receiver_category`, because the two use different vocabularies.


**Visualizations I am interested in:** a monthly trend with both datasets, a log-log country scatter plot, a country-by-year heatmap, a group-share area chart, a side-by-side sector comparison, and a world map of victims per million.